In [8]:
all_image_paths_tf = tf.constant(all_image_paths, dtype=tf.string)
all_image_labels_tf = tf.constant(all_image_labels, dtype=tf.int32)

dataset = tf.data.Dataset.from_tensor_slices((all_image_paths_tf, all_image_labels_tf))
dataset = dataset.shuffle(buffer_size=len(all_image_paths), seed=123)

dataset_size = len(all_image_paths)
train_size = int(0.8 * dataset_size)

train_ds_raw = dataset.take(train_size)
val_ds_raw = dataset.skip(train_size)

In [9]:
def _decode_and_resize_img(img_path, label, img_size=(224, 224)):
    def _decode_image_py_func(path, lbl):
        try:
            img_bytes = tf.io.read_file(path)

            img = tf.image.decode_image(img_bytes, channels=3, expand_animations=False)
            img = tf.image.resize(img, img_size)
            img = tf.cast(img, tf.uint8)
            return img, lbl, True
        except (tf.errors.InvalidArgumentError, tf.errors.OutOfRangeError, tf.python.framework.errors_impl.DataLossError) as e:

            return tf.zeros((img_size[0], img_size[1], 3), dtype=tf.uint8), lbl, False

    image, label, success = tf.py_function(
        _decode_image_py_func,
        inp=[img_path, label],
        Tout=[tf.uint8, tf.int32, tf.bool]
    )

    image.set_shape([img_size[0], img_size[1], 3])
    label.set_shape([])
    success.set_shape([])

    return image, label, success
